# Build an AI-Powered Analytics App

A **semantic view** is a Snowflake object that describes the business meaning of your data. Think of it as a business glossary layered on top of your database -- it tells AI tools what your columns mean, how your tables relate to each other, and what metrics to calculate. Once defined, tools like Cortex Analyst can translate plain English questions into accurate SQL automatically, without the user needing to know any SQL.

Without a semantic view, an analyst would need to know that `SS_QUANTITY` means "store sales quantity" or that `STORE_SALES` joins to `CUSTOMER` on `SS_CUSTOMER_SK`. With a semantic view, cryptic column names become readable business terms (e.g., `SS_QUANTITY` → `store_sales_quantity`, `SS_NET_PROFIT` → `net_profit`), table relationships are defined once, and every downstream tool (Cortex Analyst, Streamlit chatbots, Snowflake Intelligence) benefits from both the clearer names and the business logic.

In this notebook, you will:
1. Create a semantic view over **TPC-DS** retail sample data, a standard benchmark dataset that simulates a chain of stores selling products to customers
2. Query the semantic view with SQL using the `SEMANTIC_VIEW()` function
3. Connect to **Cortex Analyst** for natural language-to-SQL
4. Build a **Streamlit in Snowflake** chatbot (code reference)

**Prerequisites**: ACCOUNTADMIN role, access to `SNOWFLAKE_SAMPLE_DATA`, and a running warehouse (e.g., `COMPUTE_WH`).

## 1. Setup

First, set up a database, schema, and views over Snowflake's built-in TPC-DS sample data. These views give you retail data (customers, items, stores, sales) to model without uploading anything.

In [ ]:
-- Check your current context
SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA();

In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
CREATE DATABASE IF NOT EXISTS SAMPLE_DATA;
USE DATABASE SAMPLE_DATA;
CREATE SCHEMA IF NOT EXISTS TPCDS_SF10TCL;
USE SCHEMA TPCDS_SF10TCL;

### Create Views from Sample Data

Semantic views cannot be created directly on shared databases like `SNOWFLAKE_SAMPLE_DATA`. To work around this, we create local views that reference the shared tables. These views act as a local layer that the semantic view can point to.

In [ ]:
CREATE OR REPLACE VIEW CUSTOMER AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.CUSTOMER;

CREATE OR REPLACE VIEW CUSTOMER_DEMOGRAPHICS AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.CUSTOMER_DEMOGRAPHICS;

CREATE OR REPLACE VIEW DATE_DIM AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.DATE_DIM;

CREATE OR REPLACE VIEW ITEM AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.ITEM;

CREATE OR REPLACE VIEW STORE AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.STORE;

CREATE OR REPLACE VIEW STORE_SALES AS
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCDS_SF10TCL.STORE_SALES;

In [ ]:
-- You should see 6 views: CUSTOMER, CUSTOMER_DEMOGRAPHICS, DATE_DIM, ITEM, STORE, STORE_SALES
SHOW VIEWS;

In [ ]:
-- Notice the cryptic column names (SS_SOLD_DATE_SK, SS_CDEMO_SK, etc.)
-- The semantic view will give these business-friendly names
SELECT * FROM STORE_SALES LIMIT 5;

## 2. Create a Semantic View

A semantic view has five sections:

1. **tables** -- which physical tables to include, with aliases and primary keys (primary keys uniquely identify each row)
2. **relationships** -- how tables join via foreign keys (a foreign key is a column in one table that points to a row in another, like a sale pointing to the customer who made it)
3. **facts** -- raw numeric values like price and quantity
4. **dimensions** -- categorical attributes for grouping and filtering (brand, state, year)
5. **metrics** -- aggregated calculations like SUM that become your KPIs

The following SQL creates the semantic view. As you read it, look for these five sections -- each one maps directly to a business concept.

In [ ]:
-- Creating semantic views requires the ACCOUNTADMIN role
USE ROLE ACCOUNTADMIN;

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW TPCDS_SEMANTIC_VIEW_SM
  tables (
    CUSTOMER primary key (C_CUSTOMER_SK),
    DATE as DATE_DIM primary key (D_DATE_SK),
    DEMO as CUSTOMER_DEMOGRAPHICS primary key (CD_DEMO_SK),
    ITEM primary key (I_ITEM_SK),
    STORE primary key (S_STORE_SK),
    STORESALES as STORE_SALES
      primary key (SS_SOLD_DATE_SK, SS_CDEMO_SK, SS_ITEM_SK, SS_STORE_SK, SS_CUSTOMER_SK)
  )
  relationships (
    SALESTOCUSTOMER as STORESALES(SS_CUSTOMER_SK) references CUSTOMER(C_CUSTOMER_SK),
    SALESTODATE as STORESALES(SS_SOLD_DATE_SK) references DATE(D_DATE_SK),
    SALESTODEMO as STORESALES(SS_CDEMO_SK) references DEMO(CD_DEMO_SK),
    SALESTOITEM as STORESALES(SS_ITEM_SK) references ITEM(I_ITEM_SK),
    SALETOSTORE as STORESALES(SS_STORE_SK) references STORE(S_STORE_SK)
  )
  facts (
    ITEM.COST as i_wholesale_cost,
    ITEM.PRICE as i_current_price,
    STORE.TAX_RATE as S_TAX_PRECENTAGE,
    STORESALES.SALES_QUANTITY as SS_QUANTITY
  )
  dimensions (
    CUSTOMER.BIRTHYEAR as C_BIRTH_YEAR,
    CUSTOMER.COUNTRY as C_BIRTH_COUNTRY,
    CUSTOMER.C_CUSTOMER_SK as c_customer_sk,
    DATE.DATE as D_DATE,
    DATE.D_DATE_SK as d_date_sk,
    DATE.MONTH as D_MOY,
    DATE.WEEK as D_WEEK_SEQ,
    DATE.YEAR as D_YEAR,
    DEMO.CD_DEMO_SK as cd_demo_sk,
    DEMO.CREDIT_RATING as CD_CREDIT_RATING,
    DEMO.MARITAL_STATUS as CD_MARITAL_STATUS,
    ITEM.BRAND as I_BRAND,
    ITEM.CATEGORY as I_CATEGORY,
    ITEM.CLASS as I_CLASS,
    ITEM.I_ITEM_SK as i_item_sk,
    STORE.MARKET as S_MARKET_ID,
    STORE.SQUAREFOOTAGE as S_FLOOR_SPACE,
    STORE.STATE as S_STATE,
    STORE.STORECOUNTRY as S_COUNTRY,
    STORE.S_STORE_SK as s_store_sk,
    STORESALES.SS_CDEMO_SK as ss_cdemo_sk,
    STORESALES.SS_CUSTOMER_SK as ss_customer_sk,
    STORESALES.SS_ITEM_SK as ss_item_sk,
    STORESALES.SS_SOLD_DATE_SK as ss_sold_date_sk,
    STORESALES.SS_STORE_SK as ss_store_sk
  )
  metrics (
    STORESALES.TOTALCOST as SUM(item.cost),
    STORESALES.TOTALSALESPRICE as SUM(SS_SALES_PRICE),
    STORESALES.TOTALSALESQUANTITY as SUM(SS_QUANTITY)
      WITH SYNONYMS = ('total sales quantity', 'total sales amount')
  )
;

In [ ]:
-- Confirm the semantic view was created
SHOW SEMANTIC VIEWS;

In [ ]:
-- The flow operator (->>) pipes DESC output into a SELECT so you can filter it
DESC SEMANTIC VIEW TPCDS_SEMANTIC_VIEW_SM
  ->> SELECT "object_kind", "property_value" as "parent_object", "object_name"
      FROM $1
      WHERE "object_kind" IN ('METRIC','DIMENSION') AND "property" IN ('TABLE');

## 3. Query Semantic Views

The `SEMANTIC_VIEW()` function lets you query a semantic view with SQL. You specify which dimensions and metrics you want, and the semantic view handles all the joins automatically -- no `JOIN` clauses needed.

In [ ]:
-- Business question: "What are the top-selling book brands in Texas in December 2002?"
SELECT * FROM SEMANTIC_VIEW (
  TPCDS_SEMANTIC_VIEW_SM
    DIMENSIONS
      Item.Brand,
      Item.Category,
      Date.Year,
      Date.Month,
      Store.State
    METRICS
      StoreSales.TotalSalesQuantity
)
WHERE Year = '2002' AND Month = '12' AND State = 'TX' AND Category = 'Books'
ORDER BY TotalSalesQuantity DESC
LIMIT 10;

In [ ]:
-- Business question: "Which item categories generated the most revenue in December 2002?"
SELECT * FROM SEMANTIC_VIEW (
  TPCDS_SEMANTIC_VIEW_SM
    DIMENSIONS
      Item.Category,
      Date.Year,
      Date.Month
    METRICS
      StoreSales.TotalSalesQuantity,
      StoreSales.TotalSalesPrice,
      StoreSales.TotalCost
)
WHERE Year = '2002' AND Month = '12'
ORDER BY TotalSalesPrice DESC
LIMIT 10;

In [ ]:
-- Business question: "What are the purchasing patterns across item categories and credit ratings in July 2002?"
SELECT * FROM SEMANTIC_VIEW (
  TPCDS_SEMANTIC_VIEW_SM
    DIMENSIONS
      Item.Category,
      Demo.Credit_Rating,
      Date.Year,
      Date.Month
    METRICS
      StoreSales.TotalSalesQuantity
)
WHERE Year = '2002' AND Month = '7'
ORDER BY TotalSalesQuantity DESC
LIMIT 20;

## 4. Cortex Analyst

Cortex Analyst is Snowflake's **text-to-SQL** engine -- it translates plain English questions into SQL queries that a database can execute. It reads your semantic view definition and generates accurate \`SEMANTIC_VIEW()\` SQL queries, so users can ask questions about data without writing any SQL themselves.

Run the cell below to generate a clickable Snowsight URL that opens Cortex Analyst.

In [ ]:
SELECT 'https://app.snowflake.com/' ||
       CURRENT_ORGANIZATION_NAME() || '/' ||
       CURRENT_ACCOUNT_NAME() ||
       '/#/cortex/analyst/databases/' ||
       CURRENT_DATABASE() ||
       '/schemas/' ||
       CURRENT_SCHEMA() ||
       '/semanticView/TPCDS_SEMANTIC_VIEW_SM' AS cortex_analyst_url;

### Try Natural Language Queries

Open the URL above in your browser. It takes you directly to the Cortex Analyst interface for your semantic view. Try these questions:

- "Show me the top selling brands in Texas in 2002"
- "What is the total sales quantity by state?"
- "Which item categories have the highest total cost?"
- "Compare sales across credit ratings for the Books category"

Cortex Analyst translates each question into a `SEMANTIC_VIEW()` SQL query and returns results.

## 5. Streamlit Chatbot

**Streamlit in Snowflake (SiS)** lets you build interactive Python web apps that run entirely inside Snowflake -- no external servers or deployment pipelines needed.

The chatbot takes a natural language question, sends it to the Cortex Analyst REST API along with your semantic view, and displays the answer with data tables and charts.

### How to create the app

1. In Snowsight, go to **Projects > Streamlit > + Streamlit App**
2. Set **App name** to `Semantic_View_Chatbot`, **Warehouse** to `COMPUTE_WH`, **Database/Schema** to `SAMPLE_DATA` / `TPCDS_SF10TCL`
3. Paste the code below into the editor

**Key functions in the code:**
- `main()` -- Entry point that sets up the page and renders the chat
- `process_user_input()` -- Sends the user's question to Cortex Analyst
- `get_analyst_response()` -- Calls the REST API with `"semantic_view"` pointing to your semantic view
- `display_sql_query()` -- Shows the generated SQL, runs it, and displays results with charts

```python
"""
Semantic View Chatbot
A conversational analytics app powered by Cortex Analyst and Semantic Views.
"""
import json
import time
from typing import Dict, List, Optional, Tuple, Union

import _snowflake
import pandas as pd
import streamlit as st
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.exceptions import SnowparkSQLException

# ---- CONFIGURATION ----
# Update this to match your semantic view's fully qualified name
SEMANTIC_VIEW = "SAMPLE_DATA.TPCDS_SF10TCL.TPCDS_SEMANTIC_VIEW_SM"
API_ENDPOINT = "/api/v2/cortex/analyst/message"
API_TIMEOUT = 50000  # milliseconds

session = get_active_session()


def main():
    if "messages" not in st.session_state:
        reset_session_state()
    show_header()
    if len(st.session_state.messages) == 0:
        process_user_input("What questions can I ask?")
    display_conversation()
    handle_user_inputs()


def reset_session_state():
    st.session_state.messages = []
    st.session_state.active_suggestion = None


def show_header():
    st.title("Semantic View Chatbot")
    st.markdown(
        "Ask questions about retail sales data in natural language. "
        "Powered by **Cortex Analyst** and **Semantic Views**."
    )
    with st.sidebar:
        st.markdown(f"**Semantic View**: `{SEMANTIC_VIEW}`")
        st.divider()
        if st.button("Clear Chat", use_container_width=True):
            reset_session_state()
            st.rerun()


def handle_user_inputs():
    user_input = st.chat_input("Ask a question about your data...")
    if user_input:
        process_user_input(user_input)
    elif st.session_state.active_suggestion is not None:
        suggestion = st.session_state.active_suggestion
        st.session_state.active_suggestion = None
        process_user_input(suggestion)


def process_user_input(prompt: str):
    new_user_message = {
        "role": "user",
        "content": [{"type": "text", "text": prompt}],
    }
    st.session_state.messages.append(new_user_message)

    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("analyst"):
        with st.spinner("Thinking..."):
            response, error_msg = get_analyst_response(st.session_state.messages)
            if error_msg is None:
                analyst_message = {
                    "role": "analyst",
                    "content": response["message"]["content"],
                    "request_id": response["request_id"],
                }
            else:
                analyst_message = {
                    "role": "analyst",
                    "content": [{"type": "text", "text": error_msg}],
                    "request_id": response.get("request_id", "unknown"),
                }
            st.session_state.messages.append(analyst_message)
            st.rerun()


def get_analyst_response(messages: List[Dict]) -> Tuple[Dict, Optional[str]]:
    request_body = {
        "messages": messages,
        "semantic_view": SEMANTIC_VIEW,
    }
    resp = _snowflake.send_snow_api_request(
        "POST", API_ENDPOINT, {}, {}, request_body, None, API_TIMEOUT,
    )
    parsed_content = json.loads(resp["content"])
    if resp["status"] < 400:
        return parsed_content, None
    else:
        error_msg = (
            f"API Error (status {resp['status']}): "
            f"{parsed_content.get('message', 'Unknown error')}"
        )
        return parsed_content, error_msg


def display_conversation():
    for idx, message in enumerate(st.session_state.messages):
        role = message["role"]
        content = message["content"]
        with st.chat_message(role):
            display_message(content, idx)


def display_message(content: List[Dict], message_index: int):
    for item in content:
        if item["type"] == "text":
            st.markdown(item["text"])
        elif item["type"] == "suggestions":
            for i, suggestion in enumerate(item["suggestions"]):
                if st.button(suggestion, key=f"sug_{message_index}_{i}"):
                    st.session_state.active_suggestion = suggestion
        elif item["type"] == "sql":
            display_sql_query(item["statement"], message_index)


def display_sql_query(sql: str, message_index: int):
    with st.expander("SQL Query", expanded=False):
        st.code(sql, language="sql")
    with st.expander("Results", expanded=True):
        with st.spinner("Running SQL..."):
            try:
                df = session.sql(sql).to_pandas()
                if df.empty:
                    st.write("Query returned no data.")
                else:
                    st.dataframe(df, use_container_width=True)
                    if len(df.columns) >= 2:
                        display_chart(df, message_index)
            except SnowparkSQLException as e:
                st.error(f"SQL execution error: {e}")


def display_chart(df: pd.DataFrame, message_index: int):
    col1, col2 = st.columns(2)
    all_cols = list(df.columns)
    x_col = col1.selectbox("X axis", all_cols, key=f"x_{message_index}")
    y_options = [c for c in all_cols if c != x_col]
    y_col = col2.selectbox("Y axis", y_options, key=f"y_{message_index}")
    try:
        st.bar_chart(df.set_index(x_col)[y_col])
    except Exception:
        st.info("Could not render chart for this data.")


if __name__ == "__main__":
    main()
```

### Test the Chatbot

Try these questions:
- "What are the top 10 brands by total sales quantity?"
- "Show me sales by state for 2002"
- "Which item categories generate the most revenue?"
- "Compare married vs single customer spending"

Follow-up questions work too (e.g., "Now filter to just Texas") because the full conversation history is sent to Cortex Analyst.

## 6. Cleanup (Optional)

Run the cells below only if you want to remove all objects created in this notebook. The underlying `SNOWFLAKE_SAMPLE_DATA` shared database is untouched.

In [ ]:
DROP SEMANTIC VIEW IF EXISTS TPCDS_SEMANTIC_VIEW_SM;

In [ ]:
DROP VIEW IF EXISTS CUSTOMER;
DROP VIEW IF EXISTS CUSTOMER_DEMOGRAPHICS;
DROP VIEW IF EXISTS DATE_DIM;
DROP VIEW IF EXISTS ITEM;
DROP VIEW IF EXISTS STORE;
DROP VIEW IF EXISTS STORE_SALES;

In [ ]:
-- Only run if you created these specifically for this tutorial
DROP SCHEMA IF EXISTS TPCDS_SF10TCL;
DROP DATABASE IF EXISTS SAMPLE_DATA;

## Key Takeaways

Congratulations! You built a conversational analytics solution from raw data to a working chatbot.

- The **semantic view** is the foundation. Define your data well once, and every downstream tool benefits.
- **Cortex Analyst** reads the semantic view and translates plain English into accurate SQL. It powers both the Streamlit chatbot and the built-in Snowsight UI.

### Why This Matters for Data Analysts

This workflow removes the biggest bottleneck in analytics: translating business questions into code. Instead of writing complex multi-table JOINs, you define the business model once in a semantic view and let AI handle the rest. Non-technical users can then ask questions in plain English through the chatbot, reducing the back-and-forth between analysts and stakeholders.

### Next Steps

The semantic view you built here also works with **Snowflake Intelligence**. By adding a Cortex Agent on top of Cortex Analyst, you can create an organization-wide AI assistant that answers data questions across multiple semantic views and data sources.

### Related Resources

- [CREATE SEMANTIC VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-semantic-view)
- [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)
- [Streamlit in Snowflake](https://docs.snowflake.com/en/developer-guide/streamlit/about-streamlit)
- [SEMANTIC_VIEW Function](https://docs.snowflake.com/en/sql-reference/functions/semantic_view)